# NLP Activities: App Review Classification

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

# ---------- ENTER YOUR FILE PATH HERE ----------
DATA_PATH = "app_store_reviews.csv"
# -------------------------------------------------

TEXT_COL = "review"
NEG      = "negative"

df = pd.read_csv(DATA_PATH, sep=";")      # this file uses ; as separator
df = df.drop_duplicates(subset=TEXT_COL)   # same review text must not be in train and test

# the 'label' column is empty, so make sentiment from the star rating
def to_sentiment(star):
    if star <= 2:
        return "negative"
    elif star == 3:
        return "neutral"
    return "positive"

df["sentiment"] = df["star"].apply(to_sentiment)
LABEL_COL = "sentiment"
df = df[[TEXT_COL, LABEL_COL]].dropna()

X_train, X_test, y_train, y_test = train_test_split(
    df[TEXT_COL], df[LABEL_COL], test_size=0.2, stratify=df[LABEL_COL], random_state=42)
print(df.shape)

## Activity 1: Pipeline (TF-IDF + Logistic Regression)

In [ ]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)

p, r, f, _ = precision_recall_fscore_support(y_test, pred, average="weighted", zero_division=0)
print("Accuracy :", round(accuracy_score(y_test, pred), 3))
print("Precision:", round(p, 3), "(weighted)")
print("Recall   :", round(r, 3), "(weighted)")
print("F1-score :", round(f, 3), "(weighted)")
print()
print(classification_report(y_test, pred, zero_division=0))

## Activity 2: Confusion matrix

In [ ]:
labels = sorted(df[LABEL_COL].unique())
cm = confusion_matrix(y_test, pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

## Activity 3: Most confident wrong predictions

In [ ]:
proba = model.predict_proba(X_test)
wrong = (pred != y_test.values)

errors = pd.DataFrame({
    "text": X_test.values[wrong],
    "true_label": y_test.values[wrong],
    "predicted_label": pred[wrong],
    "predicted_probability": proba[wrong].max(axis=1),
})
errors = errors.sort_values("predicted_probability", ascending=False)
errors.head(5)

## Activity 4: Class distribution and class balancing

In [ ]:
print(df[LABEL_COL].value_counts())
print()
print(df[LABEL_COL].value_counts(normalize=True).round(3))

In [ ]:
balanced = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
balanced.fit(X_train, y_train)
pred_bal = balanced.predict(X_test)

def neg_scores(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[NEG], zero_division=0)
    return [round(p[0], 3), round(r[0], 3), round(f[0], 3)]

comparison = pd.DataFrame(
    {"Original": neg_scores(y_test, pred),
     "Balanced": neg_scores(y_test, pred_bal)},
    index=["Precision", "Recall", "F1"])
print("Negative class comparison")
comparison

## Activity 5: Flag negative reviews

In [ ]:
# paste the snippets given in the assignment here
new_reviews = [
    "App keeps crashing every time I open it",
    "Love the new design, very smooth",
    "Worst update ever, I want a refund",
    "It works fine, nothing special",
]

new_pred = model.predict(new_reviews)
result = pd.DataFrame({"review": new_reviews, "prediction": new_pred})
result["flagged_negative"] = result["prediction"] == NEG
print(result)

print("\nFlagged as negative:")
for t in result[result["flagged_negative"]]["review"]:
    print("-", t)

p, r, f, _ = precision_recall_fscore_support(y_test, pred, labels=[NEG], zero_division=0)
print(f"\nNegative class -> precision: {p[0]:.2f}, recall: {r[0]:.2f}")